# VisA PCB1~PCB4의 EfficientNet-B0 교차 검증

같은 원본 분할에서 **기본 패치**와 **추가 결함 패치 + 오버샘플링**을 비교한다. 정상은 0, 불량은 1이다. 원본 점수는 기본 격자의 모든 타일 중 가장 높은 불량 확률이다.

`deeppcb`에서 `python src3/07-1_train_visa.py`로 결과를 생성한 뒤 이 노트북을 위에서부터 실행한다. 결과 폴더는 `src3/artifacts_visa_kfold`다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import MaxNLocator, PercentFormatter

start = Path.cwd().resolve()
locations = [parent / suffix for parent in (start, *start.parents)
             for suffix in ("", "src3", "deeppcb/src3", "DeepPCB/deeppcb/src3",
                            "data_example/DeepPCB/deeppcb/src3")]
src3 = next((folder for folder in locations if (folder / "07-1_train_visa.py").is_file()), None)
if src3 is None:
    raise FileNotFoundError("deeppcb 또는 src3 폴더에서 노트북을 열어 주세요.")

RESULTS = src3 / "artifacts_visa_kfold"
DETAIL_MODE = "operating"  # default: 0.5, operating: 내부 검증에서 고른 임계값
DETAIL_FOLD = 1

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8-sig"))

def table(frame):
    display(frame.style.format(precision=4, na_rep="—").hide(axis="index"))

required = ("experiment.json", "summary.csv", "paired_deltas.csv", "fold_metrics.csv",
            "run_details.csv", "fold_assignments.csv")
missing = [name for name in required if not (RESULTS / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"07-1_train_visa.py를 먼저 실행해 교차 검증을 완료하세요. {RESULTS} | 누락: {', '.join(missing)}"
    )

experiment = read_json(RESULTS / "experiment.json")
if DETAIL_MODE not in {"default", "operating"} or DETAIL_FOLD not in range(1, experiment["folds"] + 1):
    raise ValueError("DETAIL_MODE 또는 DETAIL_FOLD 설정을 확인하세요.")

summary = pd.read_csv(RESULTS / "summary.csv")
deltas = pd.read_csv(RESULTS / "paired_deltas.csv")
fold_metrics = pd.read_csv(RESULTS / "fold_metrics.csv")
details = pd.read_csv(RESULTS / "run_details.csv")
assignments = pd.read_csv(RESULTS / "fold_assignments.csv")

conditions = experiment["conditions"]
names = {"baseline": "Baseline", "patch_oversample": "Patch + oversampling"}
styles = {"baseline": ("#315580", "s"), "patch_oversample": ("#ad5018", "o")}
mode_names = {"default": "Threshold 0.5", "operating": "Validation-selected threshold"}
measures = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc", "fpr", "fnr"]
fold_dir = RESULTS / "folds" / f"fold_{DETAIL_FOLD:02d}"

## 평균 성능과 표준편차

평가 폴드별 원본 지표의 산술 평균과 표본 표준편차를 표시한다. Δ는 같은 폴드에서 ‘패치 + 오버샘플링 − 기본’의 차이를 계산한 값이며 단위는 퍼센트포인트다. FPR과 FNR은 낮을수록 좋다. 지표가 정의된 폴드 수를 함께 표시하고, 두 조건 모두 값이 있는 폴드로 Δ를 계산한다.

In [ ]:
training = experiment["config"]["training"]
display(Markdown(
    f"**{experiment['folds']}-Fold · 학습률 {training['learning_rate']:g} · "
    f"최대 {training['epochs']}에폭 · 조기 종료 patience {training['early_stopping_patience']}**"
))

def mean_std(frame, scale=1):
    return [f"{row.mean * scale:.3f} ± {row.std * scale:.3f} ({int(row.folds)})"
            if pd.notna(row.std) else
            f"{row.mean * scale:.3f} (표준편차 —, {int(row.folds)})"
            if pd.notna(row.mean) else "— (0)"
            for row in frame.itertuples()]

overall = summary.loc[summary["scope"] == "all"]
paired_overall = deltas.loc[deltas["scope"] == "all"]
for mode, title in mode_names.items():
    display(Markdown(f"### {title}"))
    view = pd.DataFrame({"지표": measures})
    for condition in conditions:
        data = overall.loc[(overall["mode"] == mode) & (overall["condition"] == condition)]
        view[names[condition]] = mean_std(data.set_index("metric").loc[measures])
    paired = paired_overall.loc[paired_overall["mode"] == mode].set_index("metric").loc[measures]
    view["Δ 평균 ± 표준편차 (pp)"] = mean_std(paired, scale=100)
    table(view)

In [ ]:
shown_metrics = ["f1", "recall", "fpr"]
positions = np.arange(len(shown_metrics))
fig, axes = plt.subplots(1, 2, figsize=(8.7, 3.7), sharey=True, layout="constrained")
for ax, (mode, title) in zip(axes, mode_names.items()):
    for offset, condition in zip((-0.12, 0.12), conditions):
        view = overall.loc[(overall["mode"] == mode) & (overall["condition"] == condition)]
        view = view.set_index("metric").loc[shown_metrics]
        color, marker = styles[condition]
        ax.plot(positions + offset, view["mean"], marker=marker, color=color,
                linestyle="none", label=names[condition])
        defined = view["std"].notna().to_numpy()
        ax.errorbar((positions + offset)[defined], view["mean"][defined], yerr=view["std"][defined],
                    fmt="none", color=color, capsize=4)
    ax.set(xticks=positions, xticklabels=["F1", "Recall", "FPR"], title=title)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Fold average ± sample SD")
axes[1].legend(loc="center right", fontsize=8)
plt.show()

## 원본 분할과 데이터 증강

기존 학습·검증 원본을 PCB 종류와 라벨로 층화해 5개 폴드로 나눈다. 같은 파일 해시의 원본은 같은 역할에 배정하고, 타일은 원본의 배정을 따른다. 각 회차의 평가 폴드를 떼어 둔 뒤, 나머지에서 약 15%를 체크포인트와 임계값 선택에 사용한다. 기존 test 분할은 교차 검증에서 제외한다.

증강 조건에서는 내부 학습 불량 원본의 겹침을 늘려 기본 격자에 없는 결함 패치를 추가한다. 불량 타일마다 정상 타일의 2배 샘플링 가중치를 주고 복원 추출한다. 두 조건의 에폭당 추출 수와 원래 학습 타일에서 계산한 클래스 가중 손실은 동일하다.

In [ ]:
base_overlap = experiment["config"]["tiling"]["overlap"]
print(f"기본 겹침 {base_overlap}px · 추가 패치 겹침 {experiment['patch_overlap']}px · "
      f"기존 test 원본 {experiment['excluded_test_originals']:,}장 별도 유지")
counts = (assignments.groupby(["fold", "role", "label"])["image_id"].size()
          .unstack("label", fill_value=0).rename(columns={0: "정상", 1: "불량"}).reset_index())
table(counts)
table(details[["fold", "condition", "train_normal_tiles", "train_defect_tiles",
               "extra_defect_patches", "draws_per_epoch"]].replace({"condition": names}))

## 학습 곡선과 최적 체크포인트

선택한 폴드에서 내부 검증 원본의 로그 손실이 가장 낮은 에폭을 표시한다. 학습 타일 손실은 두 조건의 샘플링 구성이 다르므로, 같은 내부 검증 원본에서 측정한 손실을 함께 확인한다.

In [ ]:
fold_details = details.loc[details["fold"] == DETAIL_FOLD].set_index("condition").loc[conditions]
display(Markdown(f"**Fold {DETAIL_FOLD}**"))
table(fold_details.reset_index()[["condition", "epochs_run", "best_epoch",
                                  "best_validation_image_log_loss"]].replace({"condition": names}))
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.2), layout="constrained")
for condition in conditions:
    history = pd.read_csv(fold_dir / condition / "history.csv")
    best_epoch = fold_details.loc[condition, "best_epoch"]
    color, marker = styles[condition]
    for ax, field, title in zip(axes, ["train_loss", "val_image_loss"],
                                ["Train tile loss", "Validation original-image loss"]):
        ax.plot(history["epoch"], history[field], color=color, marker=marker, markersize=3,
                label=names[condition])
        ax.set(xlabel="Epoch", ylabel="Loss", title=title)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(alpha=0.18)
    axes[1].axvline(best_epoch, color=color, linestyle="--", alpha=0.65)
axes[1].legend(fontsize=8)
plt.show()

## 내부 검증에서 선택한 운영 임계값

설정의 정상 오탐률(FPR) 상한 안에서 불량 재현율(recall)을 최대화한다. 목표를 함께 충족했는지와 실제 선택값을 기록하고, 그 임계값을 해당 평가 폴드에 그대로 적용한다. 곡선과 점은 선택한 폴드의 내부 검증 결과다.

In [ ]:
policy = experiment["config"]["threshold"]
print(f"목표 recall ≥ {policy['min_recall']:.1%} · FPR ≤ {policy['max_fpr']:.1%}")
table(details[["fold", "condition", "threshold", "calibration_recall", "calibration_fpr",
               "calibration_targets_met"]].replace({"condition": names}))

fig, ax = plt.subplots(figsize=(6.9, 3.7), layout="constrained")
for condition in conditions:
    directory = fold_dir / condition
    rule = read_json(directory / "threshold.json")
    curve = pd.read_csv(directory / "threshold_curve.csv").sort_values("threshold", ascending=False)
    color, marker = styles[condition]
    ax.step(curve["fpr"], curve["recall"], where="post", color=color, label=names[condition])
    selected = rule["validation"]
    ax.scatter(selected["fpr"], selected["recall"], marker=marker, color=color, zorder=4)
    print(f"{names[condition]} · {rule['threshold']:.6f} · {rule['reason']}")
ax.axvline(policy["max_fpr"], color="#626970", linestyle="--", lw=1, label="FPR cap")
ax.axhline(policy["min_recall"], color="#626970", linestyle=":", lw=1, label="Recall target")
ax.set(xlabel="False-positive rate", ylabel="Defect recall",
       title=f"Fold {DETAIL_FOLD} | Inner validation")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(fontsize=8)
plt.show()

## 평가 폴드별 원본 성능

설정의 `DETAIL_MODE`로 판정 기준을 선택한다. 현재 기준의 폴드별 성능과 `DETAIL_FOLD`의 혼동행렬을 표시한다.

In [ ]:
view = fold_metrics.loc[(fold_metrics["scope"] == "all") & (fold_metrics["mode"] == DETAIL_MODE)]
columns = ["fold", "condition", "threshold", "n_normal", "n_defect", "fp", "fn",
           "f1", "recall", "fpr", "roc_auc", "pr_auc"]
display(Markdown(f"**{mode_names[DETAIL_MODE]}**"))
table(view[columns].replace({"condition": names}))

selected = view.loc[view["fold"] == DETAIL_FOLD].set_index("condition").loc[conditions]
largest = max(1, int(selected[["tn", "fp", "fn", "tp"]].max().max()))
fig, axes = plt.subplots(1, 2, figsize=(6.2, 3.2), layout="constrained")
for ax, (condition, row) in zip(axes, selected.iterrows()):
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]], dtype=int)
    ax.imshow(matrix, vmin=0, vmax=largest, cmap="Blues")
    for (i, j), value in np.ndenumerate(matrix):
        ax.text(j, i, str(value), ha="center", va="center",
                color="white" if value > largest * 0.52 else "#14283a")
    ax.set(xticks=[0, 1], xticklabels=["Normal", "Defect"], xlabel="Predicted",
           yticks=[0, 1], yticklabels=["Normal", "Defect"], ylabel="Actual",
           title=f"{names[condition]} | t={row['threshold']:.4f}")
fig.suptitle(f"Fold {DETAIL_FOLD} | Evaluation originals")
plt.show()

### PCB 종류별 성능

PCB1~PCB4마다 재현율과 FPR의 폴드 평균, 표준편차, 지표가 정의된 폴드 수를 확인한다.

In [ ]:
board = summary.loc[summary["scope"].str.startswith("VisA/") & (summary["mode"] == DETAIL_MODE)
                    & summary["metric"].isin(["recall", "fpr"])].copy()
board["PCB"] = board["scope"].str.split("/").str[-1].str.upper()
board["조건"] = board["condition"].map(names)
board["평균 ± 표준편차 (폴드 수)"] = mean_std(board)
table(board[["PCB", "metric", "조건", "평균 ± 표준편차 (폴드 수)"]])

## 결과 해석

표준편차는 폴드 사이의 성능 차이를 나타낸다. 각 폴드의 운영 임계값은 해당 모델과 내부 검증셋에 대응한다. PR-AUC는 기존 코드와 같은 precision–recall 곡선의 사다리꼴 면적을 사용한다. 패치 추가와 오버샘플링의 결합 효과를 비교하는 실험이다.